In [51]:
import numpy as np
import time

import mbuild as mb
from mbuild import Polymer
from mbuild.path import Path, cyclic, lamellar, hard_sphere_random_walk
from mbuild.path.constraints import CuboidConstraint, SphereConstraint, CylinderConstraint
from mbuild.path.termination import Termination, NumSites, NumAttempts, WithinCoordinate
from mbuild.path.bias import TargetType, TargetDirection, AvoidType, AvoidDirection, TargetCoordinate, AvoidCoordinate
from mbuild.path.points import AnglesSampler
from mbuild.exceptions import PathConvergenceError

from mbuild.simulation import HoomdSimulation, hoomd_fire, hoomd_cap_displacement, ForcesHandler

from gmso.core.forcefield import ForceField
from gmso.parameterization.parameterize import apply

gaff = ForceField("../files/gaff.xml")
oplsaa = ForceField("oplsaa")
spcfw = ForceField("../files/spcfw.xml")

In [2]:
def relax_compound(compound, cap_displacement_steps, fire_iterations, fire_steps, forcefield):
    """"""
    hoomd_sim = HoomdSimulation(compound=compound, forcefield=forcefield, r_cut=1.2)
    hoomd_cap_displacement(
        n_steps=cap_displacement_steps,
        compound=compound,
        sim=hoomd_sim,
        forces_handler=ForcesHandler(),
        max_displacement=1e-3,
        dt=1
    )
    hoomd_fire(compound=compound, sim=hoomd_sim, n_iterations=5, n_steps=500, forces_handler=ForcesHandler())

In [3]:
# Discussion of external modules for Path

# Running multiple random walks:

In [4]:
num_chains = 50
chain_lengths = 50

system = Path()
for i in range(num_chains):
    rw = hard_sphere_random_walk(
        path=system,
        termination=chain_lengths,
        radius=1,
        seed=42,
        bond_length=1.01
    )
system.visualize(radius=1)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Controlling chain angles

In [5]:
num_chains = 50
chain_lengths = 50

sample_angles = (np.pi/2, np.pi/1.8) # Sampling uniformly from a small range of small angles

sample_angles = (np.pi/1.2, np.pi) # Sampling uniformly from a small range of large angles

sample_angles = AnglesSampler("normal", dict(loc=1.6, scale=0.3)) # Sampling from normal distribution, small average

sample_angles = AnglesSampler("normal", dict(loc=2.6, scale=0.3)) # Sampling from normal distribution, large average

sample_angles = AnglesSampler("normal", dict(loc=2.3, scale=0.7)) # Sampling from normal distribution, larger std

system = Path()
for i in range(num_chains):
    rw = hard_sphere_random_walk(
        path=system,
        termination=chain_lengths,
        radius=1,
        seed=42,
        bond_length=1.01,
        rw_angles=sample_angles # Change the (min, max) angles to change chain conformations
    )
system.visualize(radius=1)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Volume Constraints

In [6]:
num_chains = 80
chain_lengths = 50

cuboid = CuboidConstraint(Lx=13, Ly=13, Lz=13, pbc=(False, False, False))
sphere = SphereConstraint(radius=8, center=(0,0,0))
cylinder = CylinderConstraint(radius=5, height=18)

system = Path()

for i in range(num_chains):
    rw = hard_sphere_random_walk(
        path=system,
        volume_constraint=cuboid,
        termination=chain_lengths,
        rw_angles=(1.5, 3.14),
        seed=45,
        radius=0.34,
        bond_length=0.341
    )

system.visualize(radius=0.34)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Multiple chain types

In [7]:
num_chains = 80
chain_lengths = 50

cuboid = CuboidConstraint(Lx=13, Ly=13, Lz=13, pbc=(False, False, False))
sphere = SphereConstraint(radius=8, center=(0,0,0))
cylinder = CylinderConstraint(radius=5, height=18)

system = Path()

for i in range(num_chains):
    if i % 2 == 0:
        name = "A"
    else:
        name = "B"
    rw = hard_sphere_random_walk(
        path=system,
        bead_name=name,
        volume_constraint=sphere,
        termination=chain_lengths,
        rw_angles=(1.5, 3.14),
        seed=45,
        radius=0.34,
        bond_length=0.341
    )

system.visualize(radius=0.34)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Packing around an mBuild Compound

In [8]:
L = 8
water_box = mb.fill_box(compound=mb.load("O", smiles=True), n_compounds=[200], box=[L, L, L])
water_box.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
cuboid = CuboidConstraint(Lx=L, Ly=L, Lz=L, center=water_box.center)

num_chains = 20
chain_lengths = 30
system = Path()

for i in range(num_chains):
    rw = hard_sphere_random_walk(
        path=system,
        include_compound=water_box,
        volume_constraint=cuboid,
        termination=chain_lengths,
        rw_angles=AnglesSampler("normal", dict(loc=2.5, scale=0.5)),
        seed=45,
        radius=0.34,
        bond_length=0.341
    )

# Convert the random walk path to a compound, so we can visualize it with the water molecules
water_box.add(system.to_compound())
water_box.visualize(bead_size=0.8)

2026-06-30 14:25:02,911 - mbuild.conversion - WARNING - No element attribute associated with '<_A pos=([0.6551 7.0976 5.5792]), 1 bonds, id: 6383101008>'; and no matching elements found based upon the compound name. Setting atomic number to zero.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Backmapping and relaxing

In [10]:
num_chains = 20
chain_lengths = 20

cuboid = CuboidConstraint(Lx=8, Ly=8, Lz=8, pbc=(False, False, False))
sphere = SphereConstraint(radius=8, center=(0,0,0))
cylinder = CylinderConstraint(radius=5, height=18)

compound = mb.Compound()

for i in range(num_chains):
    rw = hard_sphere_random_walk(
        include_compound=compound,
        volume_constraint=cuboid,
        termination=chain_lengths,
        rw_angles=(1.5, 3.14),
        seed=45+i,
        radius=0.34,
        bond_length=0.341
    )
    peg_monomer = mb.load("C{<}CO{>}", smiles=True)
    peg_chain = Polymer()
    peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
    peg_chain.build_from_path(rw)
    relax_compound(peg_chain, cap_displacement_steps=1000, fire_iterations=5, fire_steps=500, forcefield=gaff)
    compound.add(peg_chain)

compound.visualize()

2026-06-30 14:25:03,523 - gmso.external.convert_hoomd - WARNING - Neither base_units or auto_scale is provided, so default units of {'length': unyt_quantity(1, 'nm'), 'energy': unyt_quantity(1, 'kJ/mol'), 'mass': unyt_quantity(1, 'amu')} are inferred.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Random Walks From Paths

In [36]:

backbone = hard_sphere_random_walk(termination=50, radius=0.34, bond_length=0.341, rw_angles=(2.8, 3.14))

for starting_index in np.arange(5, 45, 5):
    branch = hard_sphere_random_walk(
        path=backbone,
        termination=20,
        bead_name="B",
        radius=0.34,
        bond_length=0.341,
        initial_point=int(starting_index),
        connectivity="link-linear",
        rw_angles=(2.4, 3.14)
    )
    

backbone.visualize(radius=0.34)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [63]:
ring = cyclic(spacing=0.34, N=50, bead_name="R")

for starting_index in np.arange(0, 50, 10):
    rw = hard_sphere_random_walk(
        path=ring,
        bead_name="A",
        radius=0.34,
        seed=int(starting_index),
        bond_length=0.341,
        initial_point=int(starting_index),
        termination=35,
        connectivity="link-linear",
        rw_angles=(2.5, 3.14)
    )

ring.visualize(radius=0.34)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [53]:
ring = cyclic(spacing=0.34, N=50, bead_name="R")
bias = TargetDirection(direction=(0, 0, 1), weight=0.75)

for starting_index in np.arange(0, 50, 5):
    rw = hard_sphere_random_walk(
        path=ring,
        bias=bias,
        bead_name="A",
        radius=0.34,
        seed=int(starting_index),
        bond_length=0.341,
        initial_point=int(starting_index),
        termination=35,
        connectivity="link-linear",
        rw_angles=(2.5, 3.14)
    )

ring.visualize(radius=0.34)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [62]:
ring = cyclic(spacing=0.34, N=50, bead_name="R")
bias = AvoidCoordinate(avoid_coordinate=(0,0,0), weight=0.75)

for starting_index in np.arange(0, 50, 5):
    rw = hard_sphere_random_walk(
        path=ring,
        bias=bias,
        bead_name="A",
        radius=0.34,
        seed=int(starting_index),
        bond_length=0.341,
        initial_point=int(starting_index),
        termination=35,
        connectivity="link-linear",
        rw_angles=(2.5, 3.14)
    )

ring.visualize(radius=0.34)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
n_chains = 100
chain_lengths = 50
L = 100
print(L)
box = CuboidConstraint(Lx=L, Ly=L, Lz=L, pbc=(True, True, True))
path = Path()

start = time.time()
for i in range(n_chains):
    walk_passed = False
    count = 0
    while not walk_passed:
        try:
            term = Termination([NumSites(chain_lengths), NumAttempts(chain_lengths*2)])
            init_points = box.sample_candidates(points=path.coordinates, n_candidates=200+count, buffer=0.01)
            random_walk = hard_sphere_random_walk(
                path=path,
                radius=1,
                volume_constraint=box,
                bond_length=1.01,
                initial_point=init_points[count],
                seed=20,
                termination=term,
                rw_angles=AnglesSampler("uniform", dict(low=1.5, high=np.pi)),
                trial_batch_size=500
            )
            if term.success:
                print("Finished", i+1)
                walk_passed = True
            else:
                count += 1
                if count > 50:
                    path.relax(bead_radius=0.5, steps=100000)
                    count = 0
        except PathConvergenceError:
            count += 1
            if count > 50:
                path.relax(bead_radius=0.5, steps=100000)
                count = 0

finish = time.time()

print("Finished", finish - start)

In [ ]:
path.visualize(radius=1)

In [ ]:
peg_monomer = mb.load("C{<}CO{>}", smiles=True)

n_chains = 20
chain_lengths = 50

box = CuboidConstraint(Lx=8, Ly=8, Lz=8, pbc=(False, False, False))
path = Path()
compound = mb.Compound()
seed = 20
for i in range(n_chains):
    random_walk = hard_sphere_random_walk(
        path=path,
        radius=0.35,
        volume_constraint=box,
        bond_length=0.351,
        seed=seed+i,
        termination=chain_lengths,
        rw_angles=(1.5, 3.14),
    )
    chain = Polymer()
    chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
    chain.build_from_path(
        path=Path(
            coordinates=random_walk.coordinates[
                chain_lengths*i:(chain_lengths*i)+chain_lengths,
            ]
        )
    )
    relax_compound(compound=chain, cap_displacement_steps=1000, fire_iterations=5, fire_steps=1000, forcefield=gaff)
    compound.add(chain)

In [ ]:
compound.visualize()